In [10]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated,Literal
from langchain_core.messages import HumanMessage,BaseMessage
from langgraph.checkpoint.memory import MemorySaver

In [4]:
from langgraph.graph.message import add_messages
class  ChatState(TypedDict):
    messages: Annotated[list[BaseMessage],add_messages]

In [5]:
load_dotenv()

llm=ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
)

In [6]:
def chat_node(state:ChatState):
    # take query from user
    messages=state['messages']
    # send to llm
    response=llm.invoke(messages)
    # store response in state
    return {'messages':[response]}

In [11]:
checkpointer=MemorySaver()
graph=StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot=graph.compile(checkpointer=checkpointer)

In [ ]:
initial_state={
    'messages':[HumanMessage(content="What is the capital of india?")]
}

In [ ]:
thread_id='1'
while True:
    user_message=input('Type Here: ')
    print('User: ',user_message)
    if user_message.strip().lower() in ['stop','quit','exit','bye']:
        print('AI: ',"Chatbot Exited")
        break

    config={'configurable':{'thread_id':thread_id}}
    response=chatbot.invoke({'messages':[HumanMessage(content=user_message)]},config=config)
    print("AI: ",response['messages'][-1].content)

User:  hi
AI:  Hi there! How can I help you today?
User:  what is the capital of india
AI:  The capital of India is **New Delhi**.
User:  exit
AI:  Chatbot Exited
